# FRED Data Preprocessing
Cleaning all seven raw series: filling gaps, converting units, computing the yield curve spread, and aligning to trading dates.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

In [4]:
RAW_DATA_PATH = Path('../../../data/raw/fred')
PROCESSED_DATA_PATH = Path('../../../data/processed/source_data/fred')

## Load raw data

In [5]:
col_names = ['risk_free_rate_pct', 'cpi_index', 'vix', 'treasury_10yr_pct',
             'fed_funds_rate_pct', 'unemployment_rate_pct', 'recession_flag']

raw = {}
for col in col_names:
    raw[col] = pd.read_csv(RAW_DATA_PATH / f'fred_{col}_raw.csv', index_col='date', parse_dates=True)
    print(f"{col}: {raw[col].shape}")

risk_free_rate_pct: (11702, 1)
cpi_index: (953, 1)
vix: (9527, 1)
treasury_10yr_pct: (16832, 1)
fed_funds_rate_pct: (864, 1)
unemployment_rate_pct: (942, 1)
recession_flag: (2059, 1)


## Clean daily series (risk-free rate, VIX, 10-year yield)
Forward-fill weekend/holiday gaps since these are daily-published series.

In [6]:
rf_filled = raw['risk_free_rate_pct'].asfreq('D').ffill()
rf_filled['risk_free_rate_decimal'] = rf_filled['risk_free_rate_pct'] / 100

vix_filled = raw['vix'].asfreq('D').ffill()

t10_filled = raw['treasury_10yr_pct'].asfreq('D').ffill()

print('Missing after ffill - risk-free rate:', rf_filled['risk_free_rate_pct'].isna().sum())
print('Missing after ffill - VIX:', vix_filled['vix'].isna().sum())
print('Missing after ffill - 10yr yield:', t10_filled['treasury_10yr_pct'].isna().sum())

Missing after ffill - risk-free rate: 0
Missing after ffill - VIX: 0
Missing after ffill - 10yr yield: 0


## Compute the yield curve spread
10-year yield minus 3-month yield. Negative values = inverted yield curve, historically a recession signal.

In [7]:
yield_curve = pd.DataFrame(index=rf_filled.index)
yield_curve['yield_curve_spread'] = t10_filled['treasury_10yr_pct'] - rf_filled['risk_free_rate_pct']
yield_curve['is_inverted'] = (yield_curve['yield_curve_spread'] < 0).astype(int)

print(f"Inverted on {yield_curve['is_inverted'].sum()} of {len(yield_curve)} days")
yield_curve.head(-5)

Inverted on 1803 of 16382 days


,yield_curve_spread,is_inverted
date,,
1981-09-01,-1.60,1
1981-09-02,-1.25,1
1981-09-03,-1.48,1
1981-09-04,-1.13,1
1981-09-05,-1.13,1
...,...,...
2026-06-29,0.51,0
2026-06-30,0.57,0
2026-07-01,0.63,0


## Clean monthly series (CPI, Fed funds rate, unemployment, recession flag)
Forward-fill to daily frequency so everything can eventually align with trading dates.

In [8]:
cpi_filled = raw['cpi_index'].asfreq('D').ffill()
cpi_filled['cpi_pct_change'] = cpi_filled['cpi_index'].pct_change()

fedfunds_filled = raw['fed_funds_rate_pct'].asfreq('D').ffill()

unrate_filled = raw['unemployment_rate_pct'].asfreq('D').ffill()

recession_filled = raw['recession_flag'].asfreq('D').ffill()

print('Missing after ffill - CPI:', cpi_filled['cpi_index'].isna().sum())
print('Missing after ffill - Fed funds:', fedfunds_filled['fed_funds_rate_pct'].isna().sum())
print('Missing after ffill - Unemployment:', unrate_filled['unemployment_rate_pct'].isna().sum())
print('Missing after ffill - Recession flag:', recession_filled['recession_flag'].isna().sum())

Missing after ffill - CPI: 0
Missing after ffill - Fed funds: 0
Missing after ffill - Unemployment: 0
Missing after ffill - Recession flag: 0


## Align everything to trading dates
Replace the placeholder business-day range with the actual trading calendar from the price data notebook once available.

In [9]:
# TODO: replace with actual trading dates from price data, e.g.:
# trading_dates = pd.read_csv(PROCESSED_DATA_PATH / 'prices_processed.csv', index_col='date', parse_dates=True).index

trading_dates = pd.bdate_range(start=rf_filled.index.min(), end=rf_filled.index.max())

combined = pd.DataFrame(index=trading_dates)
combined.index.name = 'date'
combined['risk_free_rate_pct'] = rf_filled.reindex(trading_dates).ffill()['risk_free_rate_pct']
combined['risk_free_rate_decimal'] = rf_filled.reindex(trading_dates).ffill()['risk_free_rate_decimal']
combined['vix'] = vix_filled.reindex(trading_dates).ffill()['vix']
combined['treasury_10yr_pct'] = t10_filled.reindex(trading_dates).ffill()['treasury_10yr_pct']
combined['yield_curve_spread'] = yield_curve.reindex(trading_dates).ffill()['yield_curve_spread']
combined['is_inverted'] = yield_curve.reindex(trading_dates).ffill()['is_inverted']
combined['cpi_index'] = cpi_filled.reindex(trading_dates).ffill()['cpi_index']
combined['cpi_pct_change'] = cpi_filled.reindex(trading_dates).ffill()['cpi_pct_change']
combined['fed_funds_rate_pct'] = fedfunds_filled.reindex(trading_dates).ffill()['fed_funds_rate_pct']
combined['unemployment_rate_pct'] = unrate_filled.reindex(trading_dates).ffill()['unemployment_rate_pct']
combined['recession_flag'] = recession_filled.reindex(trading_dates).ffill()['recession_flag']

print(f"Combined shape: {combined.shape}")
combined.head(-5)

Combined shape: (11702, 11)


,risk_free_rate_pct,risk_free_rate_decimal,vix,treasury_10yr_pct,yield_curve_spread,is_inverted,cpi_index,cpi_pct_change,fed_funds_rate_pct,unemployment_rate_pct,recession_flag
date,,,,,,,,,,,
1981-09-01,17.01,0.1701,NaN,15.41,-1.60,1,93.100,0.009761,15.87,7.6,1.0
1981-09-02,16.65,0.1665,NaN,15.40,-1.25,1,93.100,0.000000,15.87,7.6,1.0
1981-09-03,16.96,0.1696,NaN,15.48,-1.48,1,93.100,0.000000,15.87,7.6,1.0
1981-09-04,16.64,0.1664,NaN,15.51,-1.13,1,93.100,0.000000,15.87,7.6,1.0
1981-09-07,16.64,0.1664,NaN,15.51,-1.13,1,93.100,0.000000,15.87,7.6,1.0
...,...,...,...,...,...,...,...,...,...,...,...
2026-06-25,3.84,0.0384,18.89,4.40,0.56,0,333.979,0.004729,3.63,4.2,0.0
2026-06-26,3.83,0.0383,18.41,4.38,0.55,0,333.979,0.004729,3.63,4.2,0.0
2026-06-29,3.87,0.0387,17.65,4.38,0.51,0,333.979,0.004729,3.63,4.2,0.0


## Save processed data
Both individually (for flexibility) and as one combined file (for convenience).

In [10]:
rf_filled.to_csv(PROCESSED_DATA_PATH / 'fred_risk_free_rate_processed.csv')
cpi_filled.to_csv(PROCESSED_DATA_PATH / 'fred_cpi_processed.csv')
vix_filled.to_csv(PROCESSED_DATA_PATH / 'fred_vix_processed.csv')
t10_filled.to_csv(PROCESSED_DATA_PATH / 'fred_treasury_10yr_processed.csv')
yield_curve.to_csv(PROCESSED_DATA_PATH / 'fred_yield_curve_processed.csv')
fedfunds_filled.to_csv(PROCESSED_DATA_PATH / 'fred_fed_funds_processed.csv')
unrate_filled.to_csv(PROCESSED_DATA_PATH / 'fred_unemployment_processed.csv')
recession_filled.to_csv(PROCESSED_DATA_PATH / 'fred_recession_flag_processed.csv')
combined.to_csv(PROCESSED_DATA_PATH / 'fred_all_series_combined.csv')

print("Saved all processed FRED data to", PROCESSED_DATA_PATH)

Saved all processed FRED data to ../../../data/processed/source_data/fred
